# ezlocalai — Google Colab Worker

Open this notebook in [Google Colab](https://colab.research.google.com/).

Set the Runtime to **Python 3** with a **T4 GPU or higher**.

Installation takes several minutes. Set an `NGROK_TOKEN` to enable tunneling.

In [ ]:
# -- Remove the system pkg_resources (broken on Python 3.12 — missing pkgutil.ImpImporter)
# -- and upgrade pip's setuptools so the fixed version is used everywhere
!apt-get remove -y python3-pkg-resources python3-setuptools -qq
!pip install -q --upgrade setuptools pip

# -- Upgrade libstdc++ to GCC 14 (required by xllamacpp — needs CXXABI_1.3.15)
!add-apt-repository -y ppa:ubuntu-toolchain-r/test
!apt-get update -qq
!apt-get install -y -qq --only-upgrade libstdc++6

# -- System dependencies
!apt-get install -y -qq \
    git build-essential cmake gcc g++ ninja-build \
    portaudio19-dev ffmpeg libportaudio2 libasound-dev \
    wget ocl-icd-opencl-dev opencl-headers \
    libclblast-dev libopenblas-dev unzip curl

# -- Clone the repo
!git clone https://github.com/DevXT-LLC/ezlocalai /content/ezlocalai
%cd /content/ezlocalai

# -- PyTorch with CUDA 12.8
!pip install -q torch==2.9.1+cu128 torchaudio==2.9.1+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

# -- Core build deps
!pip install -q 'numpy>=1.26.0' Cython 'setuptools>=78.1.1'

# -- pkuseg (required by chatterbox-tts, needs --no-build-isolation)
!pip install -q pkuseg==0.0.25 --no-build-isolation

# -- Main requirements (ignore Colab version conflicts — they are warnings only)
!pip install -q -r cuda-requirements.txt

# -- chatterbox-tts without its pinned transformers dep
!pip install -q chatterbox-tts --no-deps

# -- Reinstall diffusers from source and latest transformers after chatterbox-tts
# -- (chatterbox --no-deps can leave stale versions that lack Qwen3/Gemma3 support)
!pip install -q --upgrade transformers
!pip install -q --force-reinstall git+https://github.com/huggingface/diffusers

# -- xllamacpp CUDA 12.8 build
# Note: esp-ppq is skipped — it pulls cryptography>=46 which breaks Colab's pyopenssl
!pip install -q xllamacpp==2026.6.9538 --force-reinstall \
    --index-url https://xorbitsai.github.io/xllamacpp/whl/cu128


In [ ]:
# -- Write .env
env_content = """ROUTER_URL=https://api.ezlocal.ai
WORKER_TUNNEL=true
DEFAULT_MODEL=unsloth/Qwen3.5-0.8B-GGUF
LLM_MAX_TOKENS=1010000
N_PARALLEL=4
NGROK_TOKEN=
"""

with open("/content/ezlocalai/.env", "w") as f:
    f.write(env_content)

print(".env written:")
print(env_content)

In [ ]:
# -- Start the worker (tunnels to router automatically via WORKER_TUNNEL=true)
%cd /content/ezlocalai
!python start.py